# 전국 수요 v4 — residual head 파인튜닝 (새 BTM 체제 반영)

기존 가중치는 `train<2025`(BTM≈0)에서 학습 → residual head 가 '한낮 봉우리'를 얹어 **맑은 날 과대(+8%)·흐린 날 과소(−2%)**.
레벨(anchor/clim)은 신체제를 이미 잘 추적하므로, **인코더·cross-attention 은 동결하고 residual head(`regressor`+`weather_bypass`)만** 재학습한다.

## 데이터 역할 (소스가 다름 — 중요)
- **train/val** = `historical`(실측 기상). 창 **2024-11-23 ~ 2025-11-30**, 격자 split(`finetune_split.csv`, p=1/6, 평일·주말·공휴일·계절 균형).
- **test** = `forecast_horizon`(예보 기상) 2025-12-15~ → 이 노트북 밖에서 로컬 `serve_land_new.py`→`eval_land_new.py` 로 채점(누수 차단: train 은 2025-11-30 에서 끊음).
- **보존**: `scaler_exog.pkl`(재fit 금지)·`DMEAN/DSTD/RESID_STD/HP` 전부 그대로. `.pth` 만 새로 만든다.

## 실행 순서
1. **런타임 → 런타임 유형 변경 → GPU (T4)**.
2. 업로드 셀에서 **5개** 올리기: `model_lt.py`, `finetune_lt.py`, `land_demand_train.csv`, `finetune_split.csv`, `weights.zip`.
   - `weights.zip` = 로컬 `5. land_demand_forecaster/training/demand_lt/weights/` 폴더를 통째로 압축(안에 `best_lt_D1..15.pth`, `scaler_exog.pkl`, `metadata_lt.pkl`).
3. 설치·가드 셀 → 파인튜닝 셀(head만, GPU 기준 빠름). 끝나면 `weights_ft.zip` 자동 다운로드.
4. 로컬에서 압축 풀어 `weights_ft/` 로 두고 → forecast 기반 test 채점.

In [ ]:
!pip -q install holidays
import torch; print('CUDA', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 5개 업로드: model_lt.py, finetune_lt.py, land_demand_train.csv, finetune_split.csv, weights.zip
from google.colab import files
up = files.upload()
print(sorted(up))

In [ ]:
# ── 가중치 압축 풀기 + 디렉터리 정리 ──
import zipfile, os, glob, shutil
os.makedirs('weights', exist_ok=True)
with zipfile.ZipFile('weights.zip') as z:
    z.extractall('weights')
# zip 안에 상위 폴더가 한 겹 있으면 평탄화(best_lt_D1.pth 가 weights/ 바로 아래 오도록)
if not os.path.exists('weights/best_lt_D1.pth'):
    hit = glob.glob('weights/**/best_lt_D1.pth', recursive=True)
    assert hit, 'weights.zip 안에서 best_lt_D1.pth 를 못 찾음 — 압축 구조 확인'
    src = os.path.dirname(hit[0])
    for f in os.listdir(src):
        shutil.move(os.path.join(src, f), os.path.join('weights', f))
print('weights/ 파일:', sorted(os.listdir('weights')))

In [ ]:
# ── 버전·자산 가드 (잘못 올렸으면 여기서 멈춤) ──
import importlib, model_lt, os; importlib.reload(model_lt)
print('MODEL:', model_lt.VERSION)
assert model_lt.EXOG == ['temp_c', 'humidity', 'solar_rad'] and hasattr(model_lt, 'compute_recent_clim'), \
    f'구버전 model_lt.py (EXOG={model_lt.EXOG}). 최신(v4) 로 다시 업로드.'
import pandas as pd
cols = list(pd.read_csv('land_demand_train.csv', nrows=1).columns)
assert 'humidity_wonju' in cols and 'cap_ppa' not in cols, f'구버전 CSV 인 듯(cols={cols}).'
sp = pd.read_csv('finetune_split.csv', parse_dates=['date'])
assert set(sp['split'].unique()) == {'train', 'val'}, 'finetune_split.csv 형식 확인'
vr = (sp.split == 'val').mean()
for f in ['scaler_exog.pkl', 'metadata_lt.pkl'] + [f'best_lt_D{n}.pth' for n in range(1, 16)]:
    assert os.path.exists(f'weights/{f}'), f'weights/{f} 없음'
print(f'가드 OK · CSV {len(cols)}컬럼 · split {len(sp)}일(val {vr:.1%}) · weights 15벌+scaler+meta 확인')

In [ ]:
# ── 파인튜닝 (D1..D15, head만). 일부만: --horizons 1 8 15 ──
!python finetune_lt.py --csv land_demand_train.csv --src_w weights --split finetune_split.csv --out weights_ft --epochs 30 --lr 5e-4

In [ ]:
# ── 산출물 압축·다운로드 (weights_ft = 새 .pth + scaler·meta 동봉) ──
import shutil, json
print('val_MAPE_ft:', json.load(open('weights_ft/metadata_ft.json'))['val_MAPE_ft'])
shutil.make_archive('weights_ft', 'zip', 'weights_ft')
from google.colab import files; files.download('weights_ft.zip')

## 산출물 & 로컬 다음 단계
`weights_ft/` = 새 `best_lt_D{1..15}.pth` + (동봉) `scaler_exog.pkl`·`metadata_lt.pkl` + `metadata_ft.json`(`val_MAPE_ft`, 파인튜닝 전/후).

로컬에서 **forecast 기반 test 채점**(서빙과 동일 경로):
1. `weights_ft.zip` 풀어서 `5. land_demand_forecaster/training/demand_lt/weights_ft/` 로.
2. `serve_land_new.py` 로 `forecast_horizon`(2025-12-15~) 백테스트 → `est_horizon_land_raw` 재생성(모델명 구분).
3. `eval_land_new.py` 로 지평×계절×낮밤 MAPE/bias, **흐림 −2% / 맑음 +8%가 줄었는지**를 보정 production 대비 확인.
4. 좋으면 `weights/` 교체 → 필요 시 보정표 재적합.